# Split MP3 Into 30s Chunks With 5s Overlap

Upload or choose one MP3 file. The notebook writes 30-second MP3 chunks with 5 seconds of overlap between neighboring chunks, then creates a zip file containing all chunks.

In [ ]:
%pip install -q pydub

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

from pydub import AudioSegment

CHUNK_SECONDS = 30
OVERLAP_SECONDS = 5
OUTPUT_DIR = Path("mp3_chunks")
ZIP_PATH = Path("mp3_chunks.zip")

In [ ]:
def ensure_ffmpeg() -> None:
    if shutil.which("ffmpeg"):
        return

    if shutil.which("apt-get"):
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

    if not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg is required to read and write MP3 files.")


ensure_ffmpeg()

In [ ]:
def upload_or_choose_mp3() -> Path:
    try:
        from google.colab import files
    except ModuleNotFoundError:
        audio_path = input("Paste the path to your MP3 file: ").strip()
        if not audio_path:
            raise ValueError("No MP3 path provided.")
        return Path(audio_path).expanduser().resolve()

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    first_name = next(iter(uploaded))
    return Path(first_name).resolve()


MP3_PATH = upload_or_choose_mp3()
print(MP3_PATH)

In [ ]:
def format_timestamp(seconds: float) -> str:
    total_seconds = int(round(seconds))
    minutes, secs = divmod(total_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours:02d}h{minutes:02d}m{secs:02d}s"


def split_mp3(
    mp3_path: Path,
    output_dir: Path = OUTPUT_DIR,
    chunk_seconds: int = CHUNK_SECONDS,
    overlap_seconds: int = OVERLAP_SECONDS,
) -> list[Path]:
    if overlap_seconds >= chunk_seconds:
        raise ValueError("overlap_seconds must be less than chunk_seconds.")

    audio = AudioSegment.from_file(mp3_path, format="mp3")
    chunk_ms = int(chunk_seconds * 1000)
    step_ms = int((chunk_seconds - overlap_seconds) * 1000)
    duration_ms = len(audio)

    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    stem = mp3_path.stem.replace(" ", "_")
    chunk_paths = []

    for index, start_ms in enumerate(range(0, duration_ms, step_ms), start=1):
        end_ms = min(start_ms + chunk_ms, duration_ms)
        if end_ms <= start_ms:
            continue

        start_label = format_timestamp(start_ms / 1000)
        end_label = format_timestamp(end_ms / 1000)
        chunk_path = output_dir / f"{stem}_chunk_{index:04d}_{start_label}-{end_label}.mp3"

        audio[start_ms:end_ms].export(chunk_path, format="mp3", bitrate="192k")
        chunk_paths.append(chunk_path)

        if end_ms == duration_ms:
            break

    return chunk_paths


def zip_chunks(chunk_paths: list[Path], zip_path: Path = ZIP_PATH) -> Path:
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
        for chunk_path in chunk_paths:
            zip_file.write(chunk_path, arcname=chunk_path.name)

    return zip_path

In [ ]:
chunk_paths = split_mp3(MP3_PATH)
zip_path = zip_chunks(chunk_paths)

duration_seconds = len(AudioSegment.from_file(MP3_PATH, format="mp3")) / 1000
print(f"Input duration: {duration_seconds:.1f}s")
print(f"Created {len(chunk_paths)} chunks in {OUTPUT_DIR.resolve()}")
print(f"Created zip: {zip_path.resolve()}")

for path in chunk_paths:
    print(path)

In [ ]:
try:
    from google.colab import files
except ModuleNotFoundError:
    print(f"Download or use the zip file here: {ZIP_PATH.resolve()}")
else:
    files.download(str(ZIP_PATH))